# Colab setup: groundwater PINN

Run this notebook at the start of every Colab session. First select **Runtime → Change runtime type → GPU**.

**Where things live on Google Drive** (folder `MyDrive/pinn/`):

| Path | What |
|---|---|
| `pinn_code.zip` | the code; make it on your laptop with `python scripts/pack_for_colab.py` and upload it |
| `data/processed/` | master table + forcing cube (from M3); the raw downloads are *not* needed on Colab |
| `runs/` | every run's config, metrics and checkpoints. Saved straight to Drive, so nothing is lost if the session dies |

If the code is later pushed to GitHub, set `REPO_URL` below instead of uploading the zip.

In [ ]:
REPO_URL = ""      # e.g. "https://github.com/<you>/pinn.git"; leave empty to use pinn_code.zip
DRIVE_ROOT = "/content/drive/MyDrive/pinn"

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(f"{DRIVE_ROOT}/runs", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/data/processed", exist_ok=True)

In [ ]:
import shutil
import zipfile

CODE = "/content/pinn"
if REPO_URL:
    if os.path.exists(CODE):
        !git -C {CODE} pull
    else:
        !git clone {REPO_URL} {CODE}
else:
    shutil.rmtree(CODE, ignore_errors=True)
    with zipfile.ZipFile(f"{DRIVE_ROOT}/pinn_code.zip") as z:
        z.extractall(CODE)
%cd {CODE}

In [ ]:
# PyTorch is already installed on Colab; this adds everything else.
!pip install -q -r requirements.txt
!pip install -q -e .

In [ ]:
# Point data and run outputs at Drive (read by src/utils/config.py).
os.environ["PINN_DATA_DIR"] = f"{DRIVE_ROOT}/data"
os.environ["PINN_RUNS_DIR"] = f"{DRIVE_ROOT}/runs"
!python scripts/check_env.py
!python -m pytest -q